In [ ]:
from akd._base import ThinkingEvent, StreamingTokenEvent, CompletedEvent, ToolCallingEvent, ToolResultEvent, RunContext, HumanInputRequiredEvent, HumanResponseEvent
from akd._base.structures import RunUsage
from akd_ext.agents import EIEAgent, EIEAgentInputSchema, EIEAgentConfig
import os
import pandas as pd
from IPython.display import display


os.environ["OPENAI_API_KEY"] = ""
os.environ["EIE_MCP_KEY"] = ""

# Pricing per token (per 1M rates / 1_000_000)
MODEL_PRICING = {
    "gpt-5.2": {
        "input": 1.75 / 1_000_000,
        "cached_input": 0.175 / 1_000_000,
        "output": 14.00 / 1_000_000,
    },
    # "gpt-4o": {
    #     "input": 2.50 / 1_000_000,
    #     "cached_input": 1.25 / 1_000_000,
    #     "output": 10.00 / 1_000_000,
    # },
    # "gpt-5-nano": {
    #     "input": 0.05 / 1_000_000,
    #     "cached_input": 0.005 / 1_000_000,
    #     "output": 0.40 / 1_000_000,
    # },
    "gpt-5-mini": {
        "input": 0.25 / 1_000_000,
        "cached_input": 0.025 / 1_000_000,
        "output": 2.00 / 1_000_000,
    },
    # "gpt-4o-mini": {
    #     "input": 0.15 / 1_000_000,
    #     "cached_input": 0.075 / 1_000_000,
    #     "output": 0.60 / 1_000_000,
    # },
}

# Cumulative cost tracker — persists across cell re-runs
cost_log = []
_prev_usage = RunUsage()

def track_usage(run_context, step_label="", model="gpt-5.2"):
    """Track per-step and cumulative cost across all models."""
    global _prev_usage
    total = run_context.usage

    # Delta = current cumulative - previous cumulative
    delta_input = total.input_tokens - _prev_usage.input_tokens
    delta_output = total.output_tokens - _prev_usage.output_tokens
    delta_requests = total.requests - _prev_usage.requests
    delta_cached = (
        total.details.get("input_tokens_details.cached_tokens", 0)
        - _prev_usage.details.get("input_tokens_details.cached_tokens", 0)
    )
    delta_non_cached = delta_input - delta_cached

    row = {
        "step": len(cost_log) + 1,
        "label": step_label,
        "model": model,
        "llm_calls": delta_requests,
        "input_tokens": delta_input,
        "cached": delta_cached,
        "output_tokens": delta_output,
    }
    # Compute cost for every model from the same token counts
    for m, p in MODEL_PRICING.items():
        row[m] = (
            delta_non_cached * p["input"]
            + delta_cached * p["cached_input"]
            + delta_output * p["output"]
        )

    cost_log.append(row)

    _prev_usage = RunUsage(
        input_tokens=total.input_tokens,
        output_tokens=total.output_tokens,
        requests=total.requests,
        details=dict(total.details),
    )

    df = pd.DataFrame(cost_log)
    model_cols = list(MODEL_PRICING.keys())
    # Add totals row
    sum_cols = ["llm_calls", "input_tokens", "cached", "output_tokens"] + model_cols
    totals = df[sum_cols].sum()
    totals["step"] = ""
    totals["label"] = "TOTAL"
    totals["model"] = ""
    df = pd.concat([df, pd.DataFrame([totals])], ignore_index=True)
    # Format money columns
    for col in model_cols:
        df[col] = df[col].map("${:.5f}".format)
    return df


/Users/ppanjuli/Dev/akd-ext-eie/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [3]:
run_context = None

In [9]:
# Re-run this cell with query="your answer" to continue the conversation

# Initialize the agent
agent = EIEAgent()

query = "yes"  # change this each run

if run_context:
    run_context = RunContext.model_validate(run_context)

async for event in agent.astream(
    EIEAgentInputSchema(query=query),
    run_context=run_context,
):
    if isinstance(event, ToolCallingEvent):
        print("\nTOOL_CALL: ", event.data.tool_call.tool_name, "\n")
    if isinstance(event, ToolResultEvent):
        content = str(event.data.result.content)
        print("\nTOOL_RESULT: ", content[:200] + "..." if len(content) > 200 else content, "\n")
    if isinstance(event, ThinkingEvent):
        print(event.data.thinking_content, end="")
    if isinstance(event, StreamingTokenEvent):
        print(event.data.token, end="")
    if isinstance(event, HumanInputRequiredEvent):
        print(f"\n--- INTERRUPT: {event.data.human_input.question}")
    if isinstance(event, CompletedEvent):
        print("\n--- COMPLETED ---")

    run_context = event.run_context

# tool_state is now inside run_context.tool_state — no separate variable needed
df = track_usage(run_context, step_label=query[:50], model=agent.config.model_name)
display(df)


TOOL_CALL:  stac_search_tool 



2026-04-17 14:53:51.041 | DEBUG    | akd_ext.tools.stac_search:_arun:130 - STAC search: collection=ct-ch4-monthgrid-v2025, bbox=[-88.4731014, 30.1435843, -84.8884169, 35.0081121], datetime=2022-01-01/2022-12-31, limit=15
2026-04-17 14:53:52.182 | DEBUG    | akd_ext.tools.stac_search:_arun:160 - STAC search returned 12 items



TOOL_RESULT:  {"items":[{"id":"ct-ch4-monthgrid-v2025-202212","collection":"ct-ch4-monthgrid-v2025","datetime":"2022-12-01T00:00:00Z","asset_url":"s3://ghgc-data-store/ct-ch4-monthgrid-v2025/CTCH4_methane_emis_tota... 



2026-04-17 14:53:53.383 | INFO     | akd_ext.tools.utils:fetch_statistics_batch:203 - Fetching stats for 12 items in parallel...
2026-04-17 14:53:53.385 | INFO     | akd_ext.tools.utils:fetch_one:208 -   Starting fetch for ct-ch4-monthgrid-v2025-202212...
2026-04-17 14:53:53.386 | INFO     | akd_ext.tools.utils:fetch_one:208 -   Starting fetch for ct-ch4-monthgrid-v2025-202211...
2026-04-17 14:53:53.386 | INFO     | akd_ext.tools.utils:fetch_one:208 -   Starting fetch for ct-ch4-monthgrid-v2025-202210...
2026-04-17 14:53:53.387 | INFO     | akd_ext.tools.utils:fetch_one:208 -   Starting fetch for ct-ch4-monthgrid-v2025-202209...
2026-04-17 14:53:53.388 | INFO     | akd_ext.tools.utils:fetch_one:208 -   Starting fetch for ct-ch4-monthgrid-v2025-202208...



TOOL_CALL:  stats_tool 


TOOL_CALL:  viz_tool 



2026-04-17 14:53:53.948 | INFO     | akd_ext.tools.utils:fetch_one:216 -   Completed fetch for ct-ch4-monthgrid-v2025-202212: error=None
2026-04-17 14:53:53.948 | INFO     | akd_ext.tools.utils:fetch_one:208 -   Starting fetch for ct-ch4-monthgrid-v2025-202207...
2026-04-17 14:53:53.996 | INFO     | akd_ext.tools.utils:fetch_one:216 -   Completed fetch for ct-ch4-monthgrid-v2025-202211: error=None
2026-04-17 14:53:53.996 | INFO     | akd_ext.tools.utils:fetch_one:208 -   Starting fetch for ct-ch4-monthgrid-v2025-202206...
2026-04-17 14:53:54.029 | INFO     | akd_ext.tools.utils:fetch_one:216 -   Completed fetch for ct-ch4-monthgrid-v2025-202210: error=None
2026-04-17 14:53:54.030 | INFO     | akd_ext.tools.utils:fetch_one:208 -   Starting fetch for ct-ch4-monthgrid-v2025-202205...
2026-04-17 14:53:54.050 | INFO     | akd_ext.tools.utils:fetch_one:216 -   Completed fetch for ct-ch4-monthgrid-v2025-202209: error=None
2026-04-17 14:53:54.050 | INFO     | akd_ext.tools.utils:fetch_one:208 


TOOL_RESULT:  {"results":[{"url":"s3://ghgc-data-store/ct-ch4-monthgrid-v2025/CTCH4_methane_emis_total_202212.tif","id":"ct-ch4-monthgrid-v2025-202212","datetime":"2022-12-01T00:00:00Z","statistics":{"b1":{"min":2.... 


TOOL_RESULT:  {"items":[{"id":"ct-ch4-monthgrid-v2025-202212","datetime":"2022-12-01T00:00:00Z","tile_url":"https://earth.gov/ghgcenter/api/raster/cog/tiles/WebMercatorQuad/{z}/{x}/{y}.png?url=s3%3A%2F%2Fghgc-data-... 

Statistics retrieved. Visualization layers generated.
--- COMPLETED ---


,step,label,model,llm_calls,input_tokens,cached,output_tokens,gpt-5.2,gpt-5-mini
0,1,i want methane datasets,gpt-5.2,2.0,10037.0,4224.0,371.0,$0.01611,$0.00230
1,2,the first one,gpt-5.2,2.0,12293.0,10112.0,138.0,$0.00752,$0.00107
2,3,2022 alabama,gpt-5.2,2.0,12665.0,12160.0,65.0,$0.00392,$0.00056
3,4,yes,gpt-5.2,2.0,12981.0,12416.0,84.0,$0.00434,$0.00062
4,5,yes,gpt-5.2,3.0,29200.0,20992.0,94.0,$0.01935,$0.00276
5,,TOTAL,,11.0,77176.0,59904.0,752.0,$0.05124,$0.00732


In [6]:
# # Inspect tool_state (persisted on run_context, never sent to LLM)
run_context.model_dump()["tool_state"]

{'datetime_range': None,
 'place_result': None,
 'collections_result': {'collections': ['ct-ch4-monthgrid-v2025',
   'ct-ch4-monthgrid-v2023',
   'emit-ch4plume-v1',
   'tm54dvar-ch4flux-monthgrid-v1',
   'tm54dvar-ch4flux-mask-monthgrid-v1'],
  'matches': [{'id': 'ct-ch4-monthgrid-v2025',
    'title': 'CarbonTracker-CH₄ Isotopic Methane Inverse Fluxes v2025',
    'description': 'Surface methane (CH₄) emissions are derived from atmospheric measurements of methane and its ¹³C carbon isotope content. Different sources of methane contain different ratios of the two stable isotopologues, ¹²CH₄ and ¹³CH₄. This makes normally indistinguishable collocated sources of methane, say from agriculture and oil and gas exploration, distinguishable. The National Oceanic and Atmospheric Administration (NOAA) collects whole air samples from its global cooperative network of flasks (https://gml.noaa.gov/ccgg/about.html), which are then analyzed for methane and other trace gases. A subset of those flasks 

In [14]:
# Inspect the messages ( this is passed to llm)
run_context.model_dump()["messages"]

[{'role': 'system',
  'content': '\nROLE\nYou are Earth Insights Explorer (EIE), a server-side backend Earth science dataset discovery and descriptive statistics agent for NASA\'s VEDA STAC catalog.\n\nYou are:\n- Strictly sequential (for analysis tasks)\n- Human-confirmation gated at critical decision points\n- Fail-fast, with one controlled exception at the stats stage (partial per-item tolerance)\n- Deterministic, reproducible in regards to tool outputs, non-speculative\n- Single-collection only (v1) and per-item independent analysis\n- Not predictive and not interpretive (no causality, no policy advice, no severity labeling)\n\nOBJECTIVE\nThe agent may support lightweight dataset discovery queries (e.g., "What datasets exist for NO2 over California?") without requiring datetime or AOI inputs. In such cases, the agent may return collection candidates before enforcing the full execution pipeline.\n\nGiven a user\'s natural-language Earth science question, help them:\n1. Parse and con

## Testing with gpt-5-mini
The goal of this experiment with gpt-5-mini is to figure out if it can reliably follow the tool sequence and pass parameters correctly, for a tool-heavy agent like EIE

In [ ]:
# Initialize the agent with a smaller model
agent = EIEAgent(EIEAgentConfig(model_name='gpt-5-mini', reasoning_effort="low"))
run_context = None

In [ ]:
agent.config.model_name

In [ ]:
# Re-run this cell with query="your answer" to continue the conversation
query = "yes"

if run_context:
    run_context = RunContext.model_validate(run_context)

async for event in agent.astream(
    EIEAgentInputSchema(query=query),
    run_context=run_context,
):
    if isinstance(event, ToolCallingEvent):
        print("TOOL_CALL: ", event.data.tool_call.tool_name)
    if isinstance(event, ToolResultEvent):
        content = str(event.data.result.content)
        print("TOOL_RESULT: ", content[:200] + "..." if len(content) > 200 else content)
    if isinstance(event, ThinkingEvent):
        print(event.data.thinking_content, end="")
    if isinstance(event, StreamingTokenEvent):
        print(event.data.token, end="")
    if isinstance(event, HumanInputRequiredEvent):
        print(f"\n--- INTERRUPT: {event.data.human_input.question}")
    if isinstance(event, CompletedEvent):
        print("\n--- COMPLETED ---")

    run_context = event.run_context

In [ ]:
## Result for gpt-5-mini : Good

## Testing with gpt-5-nano
The goal of this experiment with gpt-5-nano is to figure out if it can reliably follow the tool sequence and pass parameters correctly, for a tool-heavy agent like EIE

In [ ]:
# Initialize the agent with a smaller model
agent = EIEAgent(EIEAgentConfig(model_name='gpt-5-nano', reasoning_effort="low"))
run_context = None

In [ ]:
# Re-run this cell with query="your answer" to continue the conversation
query = "yes"

if run_context:
    run_context = RunContext.model_validate(run_context)

async for event in agent.astream(
    EIEAgentInputSchema(query=query),
    run_context=run_context,
):
    if isinstance(event, ToolCallingEvent):
        print("TOOL_CALL: ", event.data.tool_call.tool_name)
    if isinstance(event, ToolResultEvent):
        content = str(event.data.result.content)
        print("TOOL_RESULT: ", content[:200] + "..." if len(content) > 200 else content)
    if isinstance(event, ThinkingEvent):
        print(event.data.thinking_content, end="")
    if isinstance(event, StreamingTokenEvent):
        print(event.data.token, end="")
    if isinstance(event, HumanInputRequiredEvent):
        print(f"\n--- INTERRUPT: {event.data.human_input.question}")
    if isinstance(event, CompletedEvent):
        print("\n--- COMPLETED ---")

    run_context = event.run_context

In [ ]:
## Result for gpt-5-nano : Okish but Weird